# GSM8K Step 1: Paper-Aligned Judging

This notebook qualifies FullKV with the eight-shot Chain-of-Thought prompt from ChunkKV Appendix G, Table 30, verifies exact token parity through the unpruned custom-cache generation path, and only then enables TDC-KV compression. Run cells in order. Do not use Run All.

## 1. Clone or update `branch-h`

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring")
URL = "https://github.com/JayGor-13/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring.git"

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--branch", "branch-h", "--single-branch", URL, str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "pull", "--ff-only", "origin", "branch-h"], cwd=REPO, check=True)

os.chdir(REPO)
REVISION = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Repository:", REPO)
print("Revision:", REVISION)

Repository: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring
Revision: ade97b6


## 2. Install and verify the environment

In [5]:
packages = [
    "transformers>=4.43,<6",
    "datasets>=5.0.1",
    "accelerate>=1.14.0",
    "pandas>=2.2",
    "pytest>=8.4",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
print("Installation complete.")

Installation complete.


In [6]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring")
URL = "https://github.com/JayGor-13/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring.git"

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--branch", "branch-h", "--single-branch", URL, str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "pull", "--ff-only", "origin", "branch-h"], cwd=REPO, check=True)

os.chdir(REPO)
REVISION = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Repository:", REPO)
print("Revision:", REVISION)

Repository: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring
Revision: ade97b6


In [7]:
test_result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_gsm8k_protocol.py",
        "tests/test_eval_metrics.py",
        "tests/test_hf_runner.py",
        "tests/test_hf_cache_e2e.py",
    ],
    cwd=REPO,
    text=True,
    timeout=1200,
)
print("Return code:", test_result.returncode)
if test_result.returncode != 0:
    raise RuntimeError("GSM8K protocol or cache parity tests failed.")

Return code: 0


## 3. Define the guarded GSM8K runner

In [8]:
import json
import time
from datetime import datetime, timezone

import pandas as pd
import torch
from IPython.display import display

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = REPO / "outputs" / "gsm8k_step1" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float32"
PROTOCOL = "chunkkv_gsm8k_8shot"
DATASET = (
    "name=gsm8k_chunkkv,source=openai/gsm8k,adapter=gsm8k,"
    "protocol=chunkkv_gsm8k_8shot,config=main,split=test,"
    "prompt_field=question,answer_field=answer"
)
print("Output directory:", OUTPUT_DIR)
print("Model dtype:", DTYPE)

def judgment_frame(payload):
    rows = []
    for run in payload.get("runs", []):
        if run.get("status") != "ok":
            continue
        judgment = run.get("judgment") or {}
        rows.append({
            "sample_id": run.get("sample_id"),
            "method": run.get("method"),
            "budget_type": run.get("config", {}).get("budget_type"),
            "budget_value": run.get("config", {}).get("budget_value"),
            "prediction_answer": judgment.get("normalized_prediction"),
            "gold_answer": judgment.get("normalized_gold"),
            "correct": judgment.get("correct"),
            "generated_tokens": len(run.get("generated_token_ids", [])),
            "prediction": run.get("prediction", ""),
            "prompt_sha256": run.get("prompt_sha256"),
        })
    return pd.DataFrame(rows)

def validate_gsm8k_payload(payload, *, require_parity=False):
    assert payload["protocols"]["gsm8k_chunkkv"]["protocol"] == PROTOCOL
    errors = [run for run in payload["runs"] if run.get("status") != "ok"]
    if errors:
        raise AssertionError(f"{len(errors)} run(s) failed: {errors[0]}")
    for run in payload["runs"]:
        assert run.get("protocol") == PROTOCOL
        assert run.get("prompt_serialization") == "chat"
        assert run.get("raw_prompt_sha256")
        assert run.get("prompt_sha256")
        assert run.get("input_token_sha256")
        assert isinstance(run.get("generated_token_ids"), list)
        assert run.get("judgment", {}).get("judge") == "gsm8k_final_numeric_exact_match"
    if require_parity:
        parity = payload["summary"]["fullkv_parity"]
        assert parity["count"] > 0
        assert parity["all_passed"] is True, parity
    return payload

def run_gsm8k(*, label, methods, budget_ratios="", max_samples=10, run_parity=False, timeout_minutes=90):
    output = OUTPUT_DIR / f"{label}.json"
    command = [
        sys.executable, "-u", "scripts/run_hf_grid.py",
        "--models", MODEL,
        "--datasets", DATASET,
        "--methods", methods,
        "--budget-ratios", budget_ratios,
        "--thetas", "0.3",
        "--recent-windows", "16",
        "--alphas", "0.6",
        "--dependency-top-k", "8",
        "--max-chunk-tokens", "64",
        "--min-budget-utilization", "0.99",
        "--max-budget-shortfall-tokens", "1",
        "--prefill-block-size", "128",
        "--tier1-score-mode", "dependency",
        "--max-samples", str(max_samples),
        "--max-length", "2048",
        "--max-new-tokens", "256",
        "--prompt-serialization", "chat",
        "--device", "cuda",
        "--dtype", DTYPE,
        "--allow-level2-fallback",
        "--progress",
        "--output", str(output),
    ]
    if run_parity:
        command.append("--require-fullkv-parity")
    print("Command:", " ".join(command))
    started = time.perf_counter()
    try:
        completed = subprocess.run(command, cwd=REPO, text=True, timeout=timeout_minutes * 60)
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f"{label} exceeded {timeout_minutes} minutes.") from exc
    print(f"Elapsed: {(time.perf_counter() - started) / 60:.1f} minutes")
    if not output.exists():
        raise RuntimeError(f"{label} did not produce {output}.")
    payload = validate_gsm8k_payload(
        json.loads(output.read_text(encoding="utf-8")),
        require_parity=run_parity,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"{label} failed with return code {completed.returncode}.")
    display(judgment_frame(payload))
    return payload

Output directory: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring/outputs/gsm8k_step1/20260804T134614Z


## 4. FullKV qualification and generation-path parity

This performs two greedy generations per sample: HuggingFace FullKV and the same unpruned cache through the custom generation path. Compression remains disabled.

In [9]:
qualification = run_gsm8k(
    label="qualification_parity_10",
    methods="fullkv",
    max_samples=10,
    run_parity=True,
    timeout_minutes=90,
)

Command: /usr/bin/python3 -u scripts/run_hf_grid.py --models Qwen/Qwen2.5-1.5B-Instruct --datasets name=gsm8k_chunkkv,source=openai/gsm8k,adapter=gsm8k,protocol=chunkkv_gsm8k_8shot,config=main,split=test,prompt_field=question,answer_field=answer --methods fullkv --budget-ratios  --thetas 0.3 --recent-windows 16 --alphas 0.6 --dependency-top-k 8 --max-chunk-tokens 64 --min-budget-utilization 0.99 --max-budget-shortfall-tokens 1 --prefill-block-size 128 --tier1-score-mode dependency --max-samples 10 --max-length 2048 --max-new-tokens 256 --prompt-serialization chat --device cuda --dtype float16 --allow-level2-fallback --progress --output /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring/outputs/gsm8k_step1/20260804T134614Z/qualification_parity_10.json --require-fullkv-parity
Elapsed: 4.6 minutes


,sample_id,method,budget_type,budget_value,prediction_answer,gold_answer,correct,generated_tokens,prediction,prompt_sha256
0,gsm8k_chunkkv_0,fullkv,fullkv,None,,18,False,256,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,e44cda47287f59e9079b99521846aafba97e50b721033b...
1,gsm8k_chunkkv_1,fullkv,fullkv,None,,3,False,256,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,06918bc759f64607c68a64707865fb1e59fcede33152c8...
2,gsm8k_chunkkv_2,fullkv,fullkv,None,,70000,False,256,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,5124e9060fb7fe1146938103df8c23f5a408b7e1b1fcba...
3,gsm8k_chunkkv_3,fullkv,fullkv,None,,540,False,256,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,50f9d5da82f4b240ca7e7d148d4a1da0a802f3d0200fba...
4,gsm8k_chunkkv_4,fullkv,fullkv,None,,20,False,256,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,1ff89ecca7736ad299261a5fb0f113f785529498bd3581...
5,gsm8k_chunkkv_5,fullkv,fullkv,None,,64,False,256,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,6708549faa856f941760449d6fc0616027f1c3adeb432a...
6,gsm8k_chunkkv_6,fullkv,fullkv,None,,260,False,256,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,5e6d757bbd771578b95f779422ac9685544d10787bd2dd...
7,gsm8k_chunkkv_7,fullkv,fullkv,None,,160,False,256,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,5dc1edc909d595ffc2003cffc0bfa4f71e8eccf9a76f58...
8,gsm8k_chunkkv_8,fullkv,fullkv,None,,45,False,256,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,b4af6634e4d56395fd8713dceb109193d4ef7f46153566...
9,gsm8k_chunkkv_9,fullkv,fullkv,None,,460,False,256,!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!...,e379576586f3e9c6033f53cda8623bf4513b2e9ab4cf4e...


In [10]:
baseline = qualification["summary"]["baseline_qa_summary"]
parity = qualification["summary"]["fullkv_parity"]
fullkv_runs = [
    run for run in qualification["runs"]
    if run.get("status") == "ok" and run.get("method") == "fullkv"
]
nonempty_answers = sum(
    bool((run.get("judgment") or {}).get("normalized_prediction"))
    for run in fullkv_runs
)
print("FullKV metric:", baseline["primary_metric"])
print("FullKV score:", baseline["primary_score"])
print("Token parity:", parity["token_parity_rate"])
print("Text parity:", parity["text_parity_rate"])
print(f"Extractable answers: {nonempty_answers}/{len(fullkv_runs)}")
assert baseline["primary_metric"] == "gsm8k_accuracy"
assert nonempty_answers == len(fullkv_runs), "Some FullKV predictions have no extractable answer."
assert baseline["primary_score"] > 0.0, "FullKV is not qualified; inspect predictions before compression."
assert parity["all_passed"] is True
print("Qualification gate: PASSED")

FullKV metric: gsm8k_accuracy
FullKV score: 0.0
Token parity: 1.0
Text parity: 1.0
Extractable answers: 0/10


AssertionError: Some FullKV predictions have no extractable answer.

In [11]:
from transformers import AutoTokenizer

fullkv_runs = [
    run for run in qualification["runs"]
    if run.get("status") == "ok" and run.get("method") == "fullkv"
]

tokenizer = AutoTokenizer.from_pretrained(MODEL)

for run in fullkv_runs[:3]:
    token_ids = run.get("generated_token_ids", [])

    print("=" * 80)
    print("Sample:", run["sample_id"])
    print("Serialization:", run.get("prompt_serialization"))
    print("Generated token count:", len(token_ids))
    print("Generated token IDs:", token_ids[:30])
    print("Tokens:", tokenizer.convert_ids_to_tokens(token_ids[:30]))
    print(
        "Decoded with special tokens:",
        repr(tokenizer.decode(token_ids, skip_special_tokens=False)),
    )
    print("Saved prediction:", repr(run.get("prediction")))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Sample: gsm8k_chunkkv_0
Serialization: chat
Generated token count: 256
Generated token IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Tokens: ['!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!', '!']
Decoded with special tokens: '!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!'
Saved prediction: '!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!'
Sample: gsm8k_chunkkv_1
Serialization: chat
Generated token count: 256
Generated token IDs: [

## 5. Ten-sample compression pilot

Run only after the qualification gate passes. This is a development curve, not the final paper-scale table.

In [ ]:
compression_pilot = run_gsm8k(
    label="tdc_compression_10",
    methods="fullkv,tdc_kv",
    budget_ratios="0.75,0.5,0.25",
    max_samples=10,
    run_parity=False,
    timeout_minutes=120,
)

In [ ]:
summary_rows = []
for group in compression_pilot["grouped_results"]:
    qa = group["qa_summary"]
    cache = group["cache_summary"]
    summary_rows.append({
        "method": group["method"],
        "budget_type": group["budget"]["type"],
        "budget_value": group["budget"]["value"],
        "samples": group["run_summary"]["successful"],
        "gsm8k_accuracy": qa["primary_score"],
        "retention": cache["avg_retention_ratio"],
        "compression_multiplier": cache["avg_compression_multiplier"],
        "budget_gap": cache["avg_budget_gap"],
    })
summary_frame = pd.DataFrame(summary_rows).sort_values(
    ["method", "budget_value"], na_position="first"
)
display(summary_frame)

## 6. Archive the evidence

In [ ]:
import shutil

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Archive:", archive)